In [ ]:
# Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def parse_conll_file(conll_file_path):
    data = []
    tokens = []
    ner_tags = []
    tag2id = {"O": 0, "Implicature": 1, "Ambiguity": 2, "Presupposition": 3, "Explicit": 4}

    with open(conll_file_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if line == "":
                if tokens:
                    data.append({'id': str(len(data)), 'ner_tags': ner_tags, 'tokens': tokens})
                    tokens, ner_tags = [], []
            else:
                parts = line.split()
                if len(parts) != 3:
                    continue
                token, _, tag = parts
                tokens.append(token)
                if tag in ["B-Implicature", "I-Implicature"]:
                    ner_tags.append(tag2id["Implicature"])
                elif tag in ["B-Ambiguity", "I-Ambiguity"]:
                    ner_tags.append(tag2id["Ambiguity"])
                elif tag in ["B-Presupposition", "I-Presupposition"]:
                    ner_tags.append(tag2id["Presupposition"])
                elif tag in ["B-Explicit", "I-Explicit"]:
                    ner_tags.append(tag2id["Explicit"])
                elif tag == "O":
                    ner_tags.append(tag2id["O"])

        if tokens:
            data.append({'id': str(len(data)), 'ner_tags': ner_tags, 'tokens': tokens})

    return data

In [ ]:
conll_file_path = "/path/to/your/file"
micro_docs = parse_conll_file(conll_file_path)

In [ ]:
# Validation: check if only expected labels are used
all_labels = {label for entry in micro_docs for label in entry["ner_tags"]}
unexpected_labels = all_labels - {0, 1, 2, 3, 4}
if unexpected_labels:
    print(f"Warning: Found unexpected label IDs: {unexpected_labels}")

In [ ]:
for entry in micro_docs:
    print(entry)

{'id': '0', 'ner_tags': [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], 'tokens': ['-DOCSTART-', 'Yes', ',', 'it', "'s", 'annoying', 'and', 'cumbersome', 'to', 'separate', 'your', 'rubbish', 'properly', 'all', 'the', 'time', '.', 'Three', 'different', 'bin', 'bags', 'stink', 'away', 'in', 'the', 'kitchen', 'and', 'have', 'to', 'be', 'sorted', 'into', 'different', 'wheelie', 'bins', '.', 'But', 'still', 'Germany', 'produces', 'way', 'too', 'much', 'rubbish', 'and', 'too', 'many', 'resources', 'are', 'lost', 'when', 'what', 'actually', 'should', 'be', 'separated', 'and', 'recycled', 'is', 'burnt', '.', 'We', 'Berliners', 'should', 'take', 'the', 'chance', 'and', 'become', 'pioneers', 'in', 'waste', 'separation', '!']}
{'id': '1', 'ner_tags': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [ ]:
!pip install transformers
!pip install accelerate -U
!pip install datasets

In [ ]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 42.3 MB/s eta 0:00:00


In [ ]:
# Mapping for conversion
id2label = {0: "O", 1: "Implicature", 2: "Ambiguity", 3: "Presupposition", 4: "Explicit"}
label2id = {"O": 0, "Implicature": 1, "Ambiguity": 2, "Presupposition": 3, "Explicit": 4}

In [ ]:
# A function to attach punctuation to sentences
import re

def reconstruct_sentence(tokens):
    sentence = ""
    for i, token in enumerate(tokens):
        if re.match(r"^[.,!?;:’”')\]]$", token):
            sentence += token
        elif token in ['(', '[', '“', '‘']:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
        else:
            if sentence and not sentence.endswith(" "):
                sentence += " "
            sentence += token
    return sentence

In [ ]:
# Creating a df with sentences instead of separate tokens
import pandas as pd

records = []
for sent in micro_docs:
    sentence_txt = reconstruct_sentence(sent["tokens"])
    label_txt = " ".join(id2label[t] for t in sent["ner_tags"])
    records.append({"sentence": sentence_txt, "ner_tag": label_txt})

df = pd.DataFrame(records)

In [ ]:
print(df.head(5))

                                            sentence  \
0  -DOCSTART- Yes, it 's annoying and cumbersome ...   
1  One can hardly move in Friedrichshain or Neukö...   
2  Health insurance companies should not cover tr...   
3  Of course there are a number of programmes in ...   
4  Intelligence services must urgently be regulat...   

                                             ner_tag  
0  O Implicature Implicature Implicature Implicat...  
1  Implicature Implicature Implicature Implicatur...  
2  Implicature Implicature Implicature Implicatur...  
3  Presupposition Presupposition Presupposition P...  
4  Explicit Explicit Explicit Explicit Explicit E...  


In [ ]:
from huggingface_hub import login

# Make sure you are logged in to Hugging Face
# You can authenticate by running: huggingface-cli login
# Or by passing your token: login(token="YOUR_TOKEN")
login()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

In [ ]:
def build_prompt(sentence):
    prompt = f"""Your task is to analyze the following text and label each sentence as *Explicit*, *Implicature*, *Ambiguity*, *Presupposition*.
Explicit refers to transparent and clearly understandable content.
Implicature refers to content that reflects universal knowledge, or that can be inferred from the utterance and its context, though not explicitly stated.
Ambiguity refers to content that may denote multiple entities or states of affairs and cannot be resolved from context alone, requiring further precision.
Presupposition refers to presenting a notion as already shared by the author and their addressee(s). Presuppositions convey information openly, assuming that it is already known and accepted.
Implicature, Ambiguity, Presupposition are subcategories of Implicit (hidden meaning).
The surrounding text is your context.

Instructions:
- For each Explicit, Implicature, Ambiguity, Presupposition found, return a separate JSON object with exactly two fields:
  - "sentence": the exact span from the input sentence expressing the argument.
  - "type": "Explicit" or "Implicature" or "Ambiguity", "Presupposition".
- If none of them is found, return one JSON object with both fields set to "".
- Do **not** wrap the JSON objects in a list (no square brackets).
- Separate multiple JSON objects with **commas and spaces only**, e.g.:
  {{ "sentence": "...", "type": "Implicature" }}, {{ "sentence": "...", "type": "Presupposition" }}
- The output must be strictly valid JSON:
  - Use double quotes only
  - Close all braces correctly
  - Do not include trailing commas
- Do not include any explanation, notes, or extra text. Output **only** the JSON objects.

Sentence:
{sentence}
"""
    return prompt

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb_config)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512, do_sample=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
predictions = []

for row in tqdm(df.itertuples(), total=len(df)):
    prompt = build_prompt(row.sentence)
    output = pipe(prompt)[0]['generated_text']

    output_clean = output.replace(prompt, "").strip()

    predictions.append({
        "sentence": row.sentence,
        "gold": row.ner_tag,
        "mistral_output": output_clean
    })

  9%|▉         | 10/112 [02:08<28:22, 16.69s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
100%|██████████| 112/112 [20:41<00:00, 11.09s/it]


In [ ]:
import json

In [ ]:
output_path = "/path/to/your/project/microtext_finegrained_predictions.json"
with open(output_path, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Saved predictions to {output_path}")

Saved predictions to /content/drive/MyDrive/Phd/Training_seq_lab_CMV/Mistral_scripts/Mistral_new/Results/microtext_finegrained_predictions.json


In [ ]:
with open("/path/to/your/project/microtext_finegrained_predictions.json") as f:
    predictions = json.load(f)

In [ ]:
import json
import nltk
from difflib import SequenceMatcher
from sklearn.metrics import classification_report
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

__________________________
__________________________
__________________________

In [ ]:
import re

def extract_mistral_sentences(mistral_output):
    mistral_output = mistral_output.strip()

    # Remove the "Output:" prefix if present
    if mistral_output.lower().startswith("output:"):
        mistral_output = mistral_output[len("output:"):].strip()

    # Extract JSON-like pairs: {"sentence": "...", "type": "..."}
    pattern = r'\{\s*"sentence"\s*:\s*"(.+?)"\s*,\s*"type"\s*:\s*"(.+?)"\s*\}'
    matches = re.findall(pattern, mistral_output, re.DOTALL)

    return [(s.strip(), t.strip().lower()) for s, t in matches]

def get_sentence_level_labels(text, gold_labels):
    tokens = word_tokenize(text.strip())  # precise tokenization
    labels = gold_labels.strip().split()
    assert len(tokens) == len(labels), f"Token and label length mismatch: {len(tokens)} vs {len(labels)}"

    sentences = sent_tokenize(text)
    sentence_labels = []

    start = 0
    for sent in sentences:
        sent_tokens = word_tokenize(sent)
        sent_labels = labels[start:start+len(sent_tokens)]
        start += len(sent_tokens)

        if "Implicature" in sent_labels:
            label = "Implicature"
        elif "Ambiguity" in sent_labels:
            label = "Ambiguity"
        elif "Presupposition" in sent_labels:
            label = "Presupposition"
        elif "Explicit" in sent_labels:
            label = "Explicit"
        else:
            label = "o"
        sentence_labels.append((sent.strip(), label))

    return sentence_labels

def best_match(pred_sent, gold_sents):
    pred_clean = pred_sent.lower()
    best_idx, best_score = -1, 0
    for i, (gold_sent, _) in enumerate(gold_sents):
        score = SequenceMatcher(None, pred_clean, gold_sent.lower()).ratio()
        if score > best_score:
            best_idx, best_score = i, score
    return best_idx if best_score > 0.6 else -1  # threshold

gold_all = []
pred_all = []

for item in predictions:
    gold_pairs = get_sentence_level_labels(item["sentence"], item["gold"])
    pred_pairs = extract_mistral_sentences(item["mistral_output"])

    matched = set()
    pred_sent_map = {}

    for pred_sent, pred_label in pred_pairs:
        idx = best_match(pred_sent, gold_pairs)
        if idx != -1 and idx not in matched:
            pred_sent_map[idx] = pred_label
            matched.add(idx)

    for i, (gold_sent, gold_label) in enumerate(gold_pairs):
      pred_label = pred_sent_map.get(i, "O")  # assign "O" if not predicted

      # Normalize both labels to have capitalized first letter
      gold_label = gold_label.strip().capitalize()
      pred_label = pred_label.strip().capitalize()

      gold_all.append(gold_label)
      pred_all.append(pred_label)

print(classification_report(
    gold_all,
    pred_all,
    labels=["Implicature", "Ambiguity", "Presupposition", "Explicit", "o"],
    digits=4
))

AssertionError: Token and label length mismatch: 85 vs 84

In [ ]:
# Choose any entry to inspect
entry = predictions[1]

gold_sentences = get_sentence_level_labels(entry["sentence"], entry["gold"])
print("\nGold sentence-level labels:")
for s, l in gold_sentences:
    print(f"- [{l.upper()}] {s}")

pred_spans = extract_mistral_sentences(entry["mistral_output"])
print("\nPredicted sentences from Mistral:")
for s, l in pred_spans:
    print(f"- [{l.upper()}] {s}")


Gold sentence-level labels:
- [IMPLICATURE] One can hardly move in Friedrichshain or Neukölln these days without permanently scanning the ground for dog dirt.
- [IMPLICATURE] And when bad luck does strike and you step into one of the many' land mines' you have to painstakingly scrape the remains off your soles.
- [PRESUPPOSITION] Higher fines are therefore the right measure against negligent, lazy or simply thoughtless dog owners.
- [IMPLICATURE] Of course, first they 'd actually need to be caught in the act by public order officers, but once they have to dig into their pockets, their laziness will sure vanish!

Predicted sentences from Mistral:
- [EXPLICIT] One can hardly move in Friedrichshain or Neukölln these days without permanently scanning the ground for dog dirt.
- [EXPLICIT] And when bad luck does strike and you step into one of the many 'land mines' you have to painstakingly scrape the remains off your soles.
- [PRESUPPOSITION] Higher fines are therefore the right measure ag

In [ ]:
# from tqdm import tqdm

# # Load predictions
# with open("/path/to/your/project/microtext_finegrained_predictions.json") as f:
#     predictions = json.load(f)

# # Inspect mismatches
# mismatch_indices = []

# for i, entry in enumerate(tqdm(predictions)):
#     tokens = word_tokenize(entry["sentence"].strip())
#     labels = entry["gold"].strip().split()

#     if len(tokens) != len(labels):
#         print(f"\n❌ Mismatch at index {i}")
#         print(f"Sentence: {entry['sentence']}")
#         print(f"Tokens ({len(tokens)}): {tokens}")
#         print(f"Labels ({len(labels)}): {labels}")
#         print(f"Difference: {len(tokens) - len(labels)} tokens")
#         mismatch_indices.append(i)

# print(f"\n🔍 Total mismatches found: {len(mismatch_indices)} out of {len(predictions)}")